# TP introduction PySpark et DBT

Comprendre comment utiliser PySpark pour transformer des données massives et DBT pour modéliser ces données dans un entrepôt.

## Nettoyer et transformer des données avec PySpark

Initialisation de la Session Spark

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/29 09:40:41 WARN Utils: Your hostname, gattano-ThinkPad-P53, resolves to a loopback address: 127.0.1.1; using 192.168.1.172 instead (on interface wlp82s0)
25/09/29 09:40:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/29 09:40:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Chargement des données depuis le CSV

In [2]:
ventes_schema = \
    'id_transaction INT, ' + \
    'client_nom STRING, ' + \
    'client_age INT, ' + \
    'client_ville STRING, ' + \
    'produit_nom STRING, ' + \
    'produit_categorie STRING, ' + \
    'produit_marque STRING, ' + \
    'prix_catalogue INT, ' + \
    'magasin_nom STRING, ' + \
    'magasin_type STRING, ' + \
    'magasin_region STRING,' + \
    'date DATE, ' + \
    'quantite INT, ' + \
    'montant_total INT'

ventes_00 = spark.read.csv('data/ventes.csv', schema=ventes_schema, header=True)

In [3]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 10)
ventes_00

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
1,Alice,25,Paris,Ordinateur,Informatique,Dell,800,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-03-12,2,NULL
2,Bob,34,Lyon,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-01-27,5,NULL
3,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,E-Shop,En ligne,National,2023-01-09,1,NULL
4,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-05-10,5,NULL
5,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,Boutique Paris,Physique,Île-de-France,2023-06-16,5,NULL
6,David,40,Bordeaux,Smartphone,Téléphonie,Apple,1200,E-Shop,En ligne,National,2023-05-31,3,NULL
7,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-04-19,3,NULL
8,Charlie,29,Marseille,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-03-28,1,NULL
9,Alice,25,Paris,Tablette,Informatique,Samsung,600,Boutique Paris,Physique,Île-de-France,2023-04-02,3,NULL
10,Emma,31,Toulouse,Casque audio,Accessoires,Sony,150,Boutique Paris,Physique,Île-de-France,2023-04-28,5,NULL


In [4]:
ventes_00.describe()

25/09/29 09:41:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,495,495,495,495,495,495,495,495,495,495,495,487,0
mean,248.0,45.0,130.16363636363636,NULL,NULL,3.0,NULL,596.2626262626262,3158.5,NULL,NULL,2.975359342915811,NULL
stddev,143.03845636750978,46.66904755831214,2191.127734403582,NULL,NULL,0.0,NULL,407.8694169120773,1678.5361678160725,NULL,NULL,1.3831022506046147,NULL
min,1,12,-29,Bordeaux,Casque audio,3,Apple,-1200,3547,En ligne,Auvergne-Rhône-Alpes,1,NULL
max,495,Emma,48781,Toulouse,Tablette,Téléphonie,Sony,1200,E-Shop,Physique,Île-de-France,5,NULL


**1. Mettre tous les champs strings en lowercase**

In [5]:
from pyspark.sql.functions import lower

ventes_01 = ventes_00. \
    withColumn('client_nom', lower(ventes_00.client_nom)). \
    withColumn('client_ville', lower(ventes_00.client_ville)). \
    withColumn('produit_nom', lower(ventes_00.produit_nom)). \
    withColumn('produit_categorie', lower(ventes_00.produit_categorie)). \
    withColumn('produit_marque', lower(ventes_00.produit_marque)). \
    withColumn('magasin_nom', lower(ventes_00.magasin_nom)). \
    withColumn('magasin_type', lower(ventes_00.magasin_type)). \
    withColumn('magasin_region', lower(ventes_00.magasin_region))
ventes_01

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
1,alice,25,paris,ordinateur,informatique,dell,800,boutique lyon,physique,auvergne-rhône-alpes,2023-03-12,2,NULL
2,bob,34,lyon,smartphone,téléphonie,apple,1200,boutique lyon,physique,auvergne-rhône-alpes,2023-01-27,5,NULL
3,alice,25,paris,montre connectée,accessoires,garmin,300,e-shop,en ligne,national,2023-01-09,1,NULL
4,alice,25,paris,smartphone,téléphonie,apple,1200,boutique paris,physique,île-de-france,2023-05-10,5,NULL
5,alice,25,paris,montre connectée,accessoires,garmin,300,boutique paris,physique,île-de-france,2023-06-16,5,NULL
6,david,40,bordeaux,smartphone,téléphonie,apple,1200,e-shop,en ligne,national,2023-05-31,3,NULL
7,alice,25,paris,smartphone,téléphonie,apple,1200,boutique lyon,physique,auvergne-rhône-alpes,2023-04-19,3,NULL
8,charlie,29,marseille,smartphone,téléphonie,apple,1200,boutique paris,physique,île-de-france,2023-03-28,1,NULL
9,alice,25,paris,tablette,informatique,samsung,600,boutique paris,physique,île-de-france,2023-04-02,3,NULL
10,emma,31,toulouse,casque audio,accessoires,sony,150,boutique paris,physique,île-de-france,2023-04-28,5,NULL


**2. Éliminer les lignes où une donnée est manquante**

Par exemple:

In [6]:
ventes_01.filter("quantite IS NULL")

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
12,emma,31,toulouse,casque audio,accessoires,sony,150,boutique lyon,physique,auvergne-rhône-alpes,2023-02-19,NULL,NULL
58,emma,31,toulouse,tablette,informatique,samsung,600,boutique paris,physique,île-de-france,2023-02-18,NULL,NULL
64,emma,31,toulouse,montre connectée,accessoires,garmin,300,boutique paris,physique,île-de-france,2023-06-08,NULL,NULL
90,alice,25,paris,smartphone,téléphonie,apple,1200,e-shop,en ligne,national,2023-03-26,NULL,NULL
110,bob,34,lyon,tablette,informatique,samsung,600,boutique lyon,physique,auvergne-rhône-alpes,2023-01-19,NULL,NULL
117,emma,31,toulouse,tablette,informatique,samsung,600,boutique lyon,physique,auvergne-rhône-alpes,2023-04-25,NULL,NULL
363,emma,31,toulouse,smartphone,téléphonie,apple,1200,e-shop,en ligne,national,2023-01-24,NULL,NULL
446,bob,34,lyon,casque audio,accessoires,sony,150,boutique lyon,physique,auvergne-rhône-alpes,2023-04-04,NULL,NULL


In [7]:
import pandas as pd

def pandas_drop_partial_entries(iterator):
    for pandas_df in iterator:
        yield pandas_df[
            (pd.notnull(pandas_df.id_transaction)) &
            (pd.notnull(pandas_df.client_nom)) &
            (pd.notnull(pandas_df.client_age)) &
            (pd.notnull(pandas_df.client_ville)) &
            (pd.notnull(pandas_df.produit_nom)) &
            (pd.notnull(pandas_df.produit_categorie)) &
            (pd.notnull(pandas_df.produit_marque)) &
            (pd.notnull(pandas_df.prix_catalogue)) &
            (pd.notnull(pandas_df.magasin_nom)) &
            (pd.notnull(pandas_df.magasin_type)) &
            (pd.notnull(pandas_df.magasin_region)) &
            (pd.notnull(pandas_df.date)) &
            (pd.notnull(pandas_df.quantite))
        ]
ventes_02 = ventes_01.mapInPandas(pandas_drop_partial_entries, schema=ventes_01.schema)

In [8]:
ventes_02.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,487,487,487,487,487,487,487,487,487,487,487,487,0
mean,249.48665297741272,45.0,131.79260780287476,NULL,NULL,3.0,NULL,596.2012320328543,3158.5,NULL,NULL,2.975359342915811,NULL
stddev,142.48610980778824,46.66904755831214,2209.050834314869,NULL,NULL,0.0,NULL,408.1612695660718,1678.5361678160725,NULL,NULL,1.3831022506046147,NULL
min,1,12,-29,bordeaux,casque audio,3,apple,-1200,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,emma,48781,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


Alternative

In [9]:
ventes_02 = ventes_01.dropna(subset=(
    'id_transaction',
    'client_nom',
    'client_age',
    'client_ville',
    'produit_nom',
    'produit_categorie',
    'produit_marque',
    'prix_catalogue',
    'magasin_nom',
    'magasin_type',
    'magasin_region',
    'date',
    'quantite'
))

In [10]:
ventes_02.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,487,487,487,487,487,487,487,487,487,487,487,487,0
mean,249.48665297741272,45.0,131.79260780287476,NULL,NULL,3.0,NULL,596.2012320328543,3158.5,NULL,NULL,2.975359342915811,NULL
stddev,142.48610980778824,46.66904755831214,2209.050834314869,NULL,NULL,0.0,NULL,408.1612695660718,1678.5361678160725,NULL,NULL,1.3831022506046147,NULL
min,1,12,-29,bordeaux,casque audio,3,apple,-1200,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,emma,48781,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**3. Éliminez les lignes où la date d'achat est extravagante**

In [14]:
ventes_02.select("date").sort("date", ascending=True)

date
1632-03-07
1702-06-28
1742-01-16
1745-04-27
1821-06-25
1852-04-14
2023-01-01
2023-01-01
2023-01-02
2023-01-03


In [15]:
ventes_02.select("date").sort("date", ascending=False)

date
2023-06-29
2023-06-29
2023-06-29
2023-06-29
2023-06-29
2023-06-28
2023-06-28
2023-06-27
2023-06-27
2023-06-27


Les dates aberrantes sont concentrées avant 2023-01-01.

In [18]:
ventes_03 = ventes_02.where(ventes_02.date > '2023-01-01')
ventes_03.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,479,479,479,479,479,479,479,479,479,479,479,479,0
mean,250.12943632567848,45.0,133.4509394572025,NULL,NULL,3.0,NULL,597.7035490605427,3158.5,NULL,NULL,2.9686847599164925,NULL
stddev,142.59450345721177,46.66904755831214,2227.4221532856905,NULL,NULL,0.0,NULL,407.7247664230635,1678.5361678160725,NULL,NULL,1.383200363506341,NULL
min,1,12,-29,bordeaux,casque audio,3,apple,-1200,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,emma,48781,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**4. Si l'âge est négatif, le mettre en positif + éliminez les entrées avec un âge aberrant**

In [19]:
ventes_03.select("client_age").sort("client_age", ascending=True)

client_age
-29
-25
25
25
25
25
25
25
25
25


In [22]:
ventes_03.select("client_age").sort("client_age", ascending=False)

client_age
48781
40
40
40
40
40
40
40
40
40


In [24]:
from pyspark.sql.functions import abs

ventes_04 = ventes_03.where(ventes_03.client_age < 100).withColumn('client_age', abs(ventes_03.client_age))
ventes_04.describe()


summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,478,478,478,478,478,478,478,478,478,478,478,478,0
mean,250.20292887029288,45.0,31.90376569037657,NULL,NULL,3.0,NULL,598.326359832636,3158.5,NULL,NULL,2.9686192468619246,NULL
stddev,142.73481382394985,46.66904755831214,4.9753213230740245,NULL,NULL,0.0,NULL,407.9237737809017,1678.5361678160725,NULL,NULL,1.3846487560295844,NULL
min,1,12,25,bordeaux,casque audio,3,apple,-1200,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,emma,40,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**5. Gérez les noms de clients aberrants**

In [25]:
ventes_04.select("client_nom").sort("client_nom", ascending=True)

client_nom
12
78
alice
alice
alice
alice
alice
alice
alice
alice


In [38]:
ventes_04.filter(ventes_04.client_nom.isin(['12', '78']))

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
146,78,31,toulouse,ordinateur,informatique,dell,800,e-shop,en ligne,national,2023-05-17,4,NULL
210,12,40,bordeaux,tablette,informatique,samsung,600,789,physique,île-de-france,2023-04-27,5,NULL


L'entrée avec le nom de client "12" a aussi un nom de magasin qui semble aberrant ("789"). On va supprimer cette entrée.
L'entrée avec le nom de client "78" semble cohérente pour le reste. On va renommer ce client en 'inconnu'.

In [44]:
from pyspark.sql.functions import udf

@udf(returnType='string')
def name_or_unknown(name: str):
    if name == '78':
        return 'inconnu'
    else:
        return name

ventes_05 = ventes_04.where(ventes_04.client_nom != '12').withColumn('client_nom', name_or_unknown(ventes_04["client_nom"]))
ventes_05.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,477,477,477,477,477,477,477,477,477,477,477,477,0
mean,250.28721174004193,NULL,31.88679245283019,NULL,NULL,3.0,NULL,598.3228511530398,3948.3333333333335,NULL,NULL,2.9643605870020964,NULL
stddev,142.87275919261208,NULL,4.966671863120142,NULL,NULL,0.0,NULL,408.3520331076049,695.1297241043095,NULL,NULL,1.382965187573422,NULL
min,1,alice,25,bordeaux,casque audio,3,apple,-1200,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,inconnu,40,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**6. Gérez les noms de ville aberrantes**

In [45]:
ventes_05.select("client_ville").distinct()

client_ville
bordeaux
paris
lyon
marseille
toulouse


Pas de ville aberrante

**7. Gérez les noms de produits aberrants**

In [46]:
ventes_05.select("produit_nom").distinct()

produit_nom
smartphone
montre connectée
ordinateur
tablette
casque audio


Pas de ville aberrante

**8. Gérez les catégories de produits aberrantes**

In [49]:
ventes_05.select("produit_nom","produit_categorie").distinct()

produit_nom,produit_categorie
smartphone,3
smartphone,téléphonie
ordinateur,3
ordinateur,informatique
montre connectée,accessoires
tablette,informatique
casque audio,accessoires


In [53]:
ventes_05.createOrReplaceTempView("ventes_05")
ventes_08 = spark.sql("""SELECT
            id_transaction,
            client_nom,
            client_age,
            client_ville,
            produit_nom,
            CASE
                WHEN produit_categorie == '3' AND produit_nom == 'smartphone' THEN "téléphonie"
                WHEN produit_categorie == '3' AND produit_nom == 'ordinateur' THEN "informatique"
                WHEN produit_categorie == '3' THEN '3'
                ELSE produit_categorie
            END AS produit_categorie,
            produit_marque,
            prix_catalogue,
            magasin_nom,
            magasin_type,
            magasin_region,
            date,
            quantite,
            montant_total
          FROM ventes_05""")
ventes_08.select("produit_nom","produit_categorie").distinct()


produit_nom,produit_categorie
smartphone,téléphonie
ordinateur,informatique
montre connectée,accessoires
tablette,informatique
casque audio,accessoires


In [54]:
ventes_08.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,477,477,477,477,477,477,477,477,477,477,477,477,0
mean,250.28721174004193,NULL,31.88679245283019,NULL,NULL,NULL,NULL,598.3228511530398,3948.3333333333335,NULL,NULL,2.9643605870020964,NULL
stddev,142.87275919261208,NULL,4.966671863120142,NULL,NULL,NULL,NULL,408.3520331076049,695.1297241043095,NULL,NULL,1.382965187573422,NULL
min,1,alice,25,bordeaux,casque audio,accessoires,apple,-1200,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,inconnu,40,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**9. Gérez les marques de produits aberrantes**

In [55]:
ventes_08.select("produit_marque").distinct()

produit_marque
apple
dell
sony
samsung
garmin


Pas de marque aberrante

**10. Gérez les prix_catalogue aberrant**

In [56]:
ventes_08.sort("prix_catalogue", ascending=True)

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
448,emma,31,toulouse,smartphone,téléphonie,apple,-1200,e-shop,en ligne,national,2023-05-22,3,NULL
20,alice,25,paris,smartphone,téléphonie,apple,-850,boutique lyon,physique,auvergne-rhône-alpes,2023-03-22,2,NULL
136,david,40,bordeaux,ordinateur,informatique,dell,-800,boutique lyon,physique,auvergne-rhône-alpes,2023-05-19,1,NULL
282,emma,31,toulouse,tablette,informatique,samsung,-600,boutique paris,physique,île-de-france,2023-06-29,1,NULL
318,alice,25,paris,tablette,informatique,samsung,-600,boutique paris,physique,île-de-france,2023-03-04,4,NULL
377,emma,31,toulouse,tablette,informatique,samsung,-600,boutique paris,physique,île-de-france,2023-06-20,1,NULL
188,charlie,29,marseille,montre connectée,accessoires,garmin,-300,boutique lyon,physique,auvergne-rhône-alpes,2023-05-26,3,NULL
374,emma,31,toulouse,montre connectée,accessoires,garmin,-300,boutique paris,physique,île-de-france,2023-06-11,5,NULL
83,charlie,29,marseille,casque audio,accessoires,sony,-150,boutique paris,physique,île-de-france,2023-04-02,2,NULL
79,bob,34,lyon,casque audio,accessoires,sony,150,boutique paris,physique,île-de-france,2023-04-24,5,NULL


In [58]:
ventes_08.select("produit_nom", "produit_marque", "prix_catalogue").distinct().sort("produit_nom")

produit_nom,produit_marque,prix_catalogue
casque audio,sony,150
casque audio,sony,-150
montre connectée,garmin,-300
montre connectée,garmin,300
ordinateur,dell,-800
ordinateur,dell,800
smartphone,apple,-1200
smartphone,apple,-850
smartphone,apple,1200
tablette,samsung,-600


Mise à part des montants qui semblent être passés en négatif, il n'y pas de valeur vraiment aberrante. Peut-être que le smartphone Apple à 850 pourrait être étudié plus précisément puisqu'il semble être une combinaison d'entrée unique dans la base de donnée...

In [59]:
from pyspark.sql.functions import abs

ventes_10 = ventes_08.withColumn('prix_catalogue', abs(ventes_08.prix_catalogue))
ventes_10.describe()


summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,477,477,477,477,477,477,477,477,477,477,477,477,0
mean,250.28721174004193,NULL,31.88679245283019,NULL,NULL,NULL,NULL,620.9643605870021,3948.3333333333335,NULL,NULL,2.9643605870020964,NULL
stddev,142.87275919261208,NULL,4.966671863120142,NULL,NULL,NULL,NULL,372.94353886472595,695.1297241043095,NULL,NULL,1.382965187573422,NULL
min,1,alice,25,bordeaux,casque audio,accessoires,apple,150,3547,en ligne,auvergne-rhône-alpes,1,NULL
max,495,inconnu,40,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**11. Gérez les informations de magasins aberrantes**

In [64]:
ventes_10.select("magasin_nom", "magasin_type", "magasin_region").distinct()

magasin_nom,magasin_type,magasin_region
boutique lyon,physique,auvergne-rhône-alpes
e-shop,en ligne,national
3547,physique,auvergne-rhône-alpes
4751,physique,île-de-france
boutique paris,physique,île-de-france


In [68]:
ventes_10.createOrReplaceTempView("ventes_10")
ventes_11 = spark.sql("""SELECT
            id_transaction,
            client_nom,
            client_age,
            client_ville,
            produit_nom,
            produit_categorie,
            produit_marque,
            prix_catalogue,
            CASE
                WHEN magasin_nom == "3547" AND magasin_type == "physique" AND magasin_region == "auvergne-rhône-alpes" THEN "boutique lyon"
                WHEN magasin_nom == "4751" AND magasin_type == "physique" AND magasin_region == "île-de-france" THEN "boutique paris"
                ELSE magasin_nom
            END AS magasin_nom,
            magasin_type,
            magasin_region,
            date,
            quantite,
            montant_total
          FROM ventes_10""")
ventes_11.select("magasin_nom","magasin_type", "magasin_region").distinct()

magasin_nom,magasin_type,magasin_region
boutique lyon,physique,auvergne-rhône-alpes
e-shop,en ligne,national
boutique paris,physique,île-de-france


In [69]:
ventes_11.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,477,477,477,477,477,477,477,477,477,477,477,477,0
mean,250.28721174004193,NULL,31.88679245283019,NULL,NULL,NULL,NULL,620.9643605870021,NULL,NULL,NULL,2.9643605870020964,NULL
stddev,142.87275919261208,NULL,4.966671863120142,NULL,NULL,NULL,NULL,372.94353886472595,NULL,NULL,NULL,1.382965187573422,NULL
min,1,alice,25,bordeaux,casque audio,accessoires,apple,150,boutique lyon,en ligne,auvergne-rhône-alpes,1,NULL
max,495,inconnu,40,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,NULL


**12. Gérez les quantités aberrantes**

In [71]:
ventes_11.select("quantite").distinct().sort("quantite", ascending=True)

quantite
1
2
3
4
5


Pas de quantité aberrante.

**13. Calculez le montant total**

In [75]:
ventes_clean = ventes_11.withColumn('montant_total', ventes_11.prix_catalogue * ventes_11.quantite)
ventes_clean.describe()

summary,id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,quantite,montant_total
count,477,477,477,477,477,477,477,477,477,477,477,477,477
mean,250.28721174004193,NULL,31.88679245283019,NULL,NULL,NULL,NULL,620.9643605870021,NULL,NULL,NULL,2.9643605870020964,1829.979035639413
stddev,142.87275919261208,NULL,4.966671863120142,NULL,NULL,NULL,NULL,372.94353886472595,NULL,NULL,NULL,1.382965187573422,1479.9722486257742
min,1,alice,25,bordeaux,casque audio,accessoires,apple,150,boutique lyon,en ligne,auvergne-rhône-alpes,1,150
max,495,inconnu,40,toulouse,tablette,téléphonie,sony,1200,e-shop,physique,île-de-france,5,6000


## Mise en base de données Postgres de la table